# 03. Data Quality — Grupo Éxito

## Objetivo

Validar la calidad, consistencia, integridad y estructura temporal
del dataset financiero de Grupo Éxito correspondiente al período
2020–2025, antes de utilizarlo en el análisis exploratorio y
modelado de series temporales.

### Validaciones

- Tipos de datos
- Valores nulos
- Registros duplicados
- Integridad temporal
- Valores financieros válidos
- Consistencia entre variables financieras
- Integridad de los períodos trimestrales

In [2]:
# Importar librerias

import pandas as pd
import numpy as np

In [3]:
# Cargar el archivo Excel en un DataFrame
df_financiero = pd.read_excel(r"C:\Users\USUARIO\Documents\GitHub\Sistema-de-Pronostico-Financiero-Grupo-Exito\data\Bronze\informe_resultados_exito.xlsx")

df_financiero.head()


,año,trimestre,fecha,ingresos,utilidad_bruta,ebitda_recurrente,resultado_neto
0,2025,1,2025-03-31,5404642,1382773,371148,93147
1,2025,2,2025-06-30,5208469,1335783,452242,146865
2,2025,3,2025-09-30,5228905,1312056,448017,142905
3,2025,4,2025-12-31,6166344,1612428,670615,209191
4,2024,1,2024-03-31,5275139,1321953,302113,-37863


In [4]:
df_financiero.shape

(24, 7)

In [5]:
# Tipos de datos de las columnas

df_financiero.dtypes

año                           int64
trimestre                     int64
fecha                datetime64[us]
ingresos                      int64
utilidad_bruta                int64
ebitda_recurrente             int64
resultado_neto                int64
dtype: object

In [6]:
# Revición de valores nulos

df_financiero.isnull().sum()

año                  0
trimestre            0
fecha                0
ingresos             0
utilidad_bruta       0
ebitda_recurrente    0
resultado_neto       0
dtype: int64

In [7]:
# Revisión de duplicados

df_financiero.duplicated().sum()

np.int64(0)

In [8]:
df_financiero.duplicated(
    subset=["año", "trimestre"]
).sum()

np.int64(0)

In [9]:
# Validación de consistencia temporal 

df_financiero.groupby("año")["trimestre"].count()

año
2020    4
2021    4
2022    4
2023    4
2024    4
2025    4
Name: trimestre, dtype: int64

In [10]:
df_financiero["fecha"].min(), df_financiero["fecha"].max()

(Timestamp('2020-03-31 00:00:00'), Timestamp('2025-12-31 00:00:00'))

### REGLAS DE CONSISTENCIA FINANCIERA

In [11]:
# Validación de valores negativos en las principales variables financieras

columnas_financieras = [
    "ingresos",
    "utilidad_bruta",
    "ebitda_recurrente"
]

for columna in columnas_financieras:
    negativos = (df_financiero[columna] < 0).sum()
    print(f"Valores negativos en {columna}: {negativos}")


Valores negativos en ingresos: 0
Valores negativos en utilidad_bruta: 0
Valores negativos en ebitda_recurrente: 0


In [12]:
# Utilidad bruta no supere ingresos

inconsistencia_utilidad_bruta = (df_financiero["utilidad_bruta"] > df_financiero["ingresos"]).sum()
print(f"Filas con inconsistencia en utilidad bruta: {inconsistencia_utilidad_bruta}")

Filas con inconsistencia en utilidad bruta: 0


In [13]:
# EBITDA recurrente no supere utilidad bruta

inconsistencia_ebitda = (df_financiero["ebitda_recurrente"] > df_financiero["utilidad_bruta"]).sum()
print(f"Filas con inconsistencia en EBITDA recurrente: {inconsistencia_ebitda}")

Filas con inconsistencia en EBITDA recurrente: 0


In [15]:
# Creación de margen bruto, margen EBITDA y margen neto como prueba de consistencia de los datos financieros

df_financiero["margen_bruto"] = (df_financiero["utilidad_bruta"] / df_financiero["ingresos"])*100

df_financiero["margen_ebitda"] = (df_financiero["ebitda_recurrente"] / df_financiero["ingresos"])*100

df_financiero["margen_neto"] = (df_financiero["resultado_neto"] / df_financiero["ingresos"])*100

In [16]:
df_financiero[
    ["margen_bruto", 
     "margen_ebitda", 
     "margen_neto"]
].round(2).describe()

,margen_bruto,margen_ebitda,margen_neto
count,24.000000,24.000000,24.000000
mean,25.541667,8.119583,1.344583
std,0.608088,1.465497,1.488670
min,24.540000,5.730000,-1.250000
25%,25.082500,6.860000,0.232500
50%,25.530000,7.945000,1.385000
75%,26.007500,8.672500,2.422500
max,26.620000,10.880000,4.060000


In [17]:
# Detección de valores atipico mediante IQR

columnas_margen = [
    "margen_bruto",
    "margen_ebitda",
    "margen_neto"
]

for columna in columnas_margen:
    
    Q1 = df_financiero[columna].quantile(0.25)
    Q3 = df_financiero[columna].quantile(0.75)
    IQR = Q3 - Q1
    
    limite_inferior = Q1 - 1.5 * IQR
    limite_superior = Q3 + 1.5 * IQR
    
    outliers = df_financiero[
        (df_financiero[columna] < limite_inferior) |
        (df_financiero[columna] > limite_superior)
    ]
    
    print(f"\n{columna}")
    print(f"Límite inferior: {limite_inferior:.2f}%")
    print(f"Límite superior: {limite_superior:.2f}%")
    print(f"Posibles valores atípicos: {len(outliers)}")


margen_bruto
Límite inferior: 23.70%
Límite superior: 27.39%
Posibles valores atípicos: 0

margen_ebitda
Límite inferior: 4.14%
Límite superior: 11.40%
Posibles valores atípicos: 0

margen_neto
Límite inferior: -3.06%
Límite superior: 5.72%
Posibles valores atípicos: 0


In [18]:
print(
    "Margen bruto negativo:",
    (df_financiero["margen_bruto"] < 0).sum()
)

print(
    "Margen EBITDA negativo:",
    (df_financiero["margen_ebitda"] < 0).sum()
)

print(
    "Margen neto negativo:",
    (df_financiero["margen_neto"] < 0).sum()
)

Margen bruto negativo: 0
Margen EBITDA negativo: 0
Margen neto negativo: 6


## Conclusión — Data Quality

El proceso de validación de calidad permitió comprobar la integridad y
consistencia del dataset financiero de Grupo Éxito para el período
2020–2025.

El conjunto de datos contiene 24 observaciones trimestrales y 7 variables.
No se identificaron valores nulos, registros duplicados ni inconsistencias
en la estructura temporal.

Las principales variables financieras presentan valores coherentes y no se
identificaron relaciones financieras incompatibles entre ingresos, utilidad
bruta y EBITDA recurrente. El resultado neto puede presentar valores
negativos, lo cual es financieramente válido y no constituye por sí mismo
un error de calidad.

Adicionalmente, el análisis de los márgenes bruto, EBITDA y neto no
identificó valores atípicos o comportamientos extraordinarios que requieran
corrección o tratamiento especial.

Por lo anterior, el dataset se considera apto para continuar con la etapa
de análisis exploratorio de series temporales.